# [프로젝트] Seq2seq 기반 한국어-영어 번역기 만들기

본 프로젝트에서는 Attention 기법을 적용한 Seq2seq 모델을 PyTorch로 구현하여 한국어를 영어로 번역하는 인공지능 번역기를 구축합니다.

---

## 목차 (Table of Contents)
1. **Step 1. 데이터 다운로드 및 환경 설정**
2. **Step 2. 데이터 정제 및 토큰화 준비**
3. **Step 3. 데이터 토큰화 및 텐서 변환**
4. **Step 4. Attention 기반 Seq2seq 모델 설계**
5. **Step 5. 모델 훈련 및 결과 시각화 (Attention Map)**
6. **회고 및 결론**

---
## Step 1. 데이터 다운로드 및 환경 설정

한영 병렬 데이터셋(`korean-english-park.train.tar.gz`)을 다운로드하고 필요한 라이브러리를 설치 및 설정합니다.

In [1]:
# 1. 한글 폰트 및 필수 패키지 설치
!sudo apt-get update -y
!sudo apt-get install -y fonts-nanum
!pip install konlpy tokenizers

# 2. Mecab 설치 (공식 스크립트 활용)
!curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh | bash

# 3. Matplotlib 폰트 캐시 강제 업데이트
import matplotlib as mpl
import matplotlib.font_manager as fm

!rm -rf ~/.cache/matplotlib
!fc-cache -fv

print("폰트 및 필수 패키지 설치가 완료되었습니다. 상단 메뉴의 [런타임] -> [세션 다시 시작] 버튼을 누른 후 코드를 다시 실행해 주세요.")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

In [2]:
import os
import re
import urllib.request
import tarfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from konlpy.tag import Mecab
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm import tqdm
import random
from collections import Counter

# 하드웨어 가속(GPU) 사용 여부 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Current Device: {device}")


PyTorch Version: 2.11.0+cu128
Current Device: cuda


In [3]:
# 시각화 시 한글 깨짐 방지를 위한 폰트 설정
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

fontpath = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
if os.path.exists(fontpath):
    fontprop = fm.FontProperties(fname=fontpath, size=12)
    plt.rcParams["font.family"] = fontprop.get_name()
    print(f"설정된 폰트: {fontprop.get_name()}")
else:
    print("Nanum 폰트를 찾을 수 없습니다. 기본 폰트를 사용합니다.")


설정된 폰트: NanumBarunGothic


In [4]:
# 데이터 다운로드 및 압축 해제
dataset_dir = os.path.expanduser("~/work/s2s_translation/datasets")
os.makedirs(dataset_dir, exist_ok=True)

tar_filepath = os.path.join(dataset_dir, "korean-english-park.train.tar.gz")

if not os.path.exists(tar_filepath):
    print("데이터 다운로드 중...")
    url = "https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz"
    urllib.request.urlretrieve(url, tar_filepath)
    print("다운로드 완료!")

# tar.gz 압축 해제
with tarfile.open(tar_filepath, "r:gz") as tar:
    tar.extractall(path=dataset_dir)
    print("압축 해제 완료!")

kor_path = os.path.join(dataset_dir, "korean-english-park.train.ko")
eng_path = os.path.join(dataset_dir, "korean-english-park.train.en")

with open(kor_path, "r", encoding="utf-8") as f:
    kor_data = f.read().splitlines()

with open(eng_path, "r", encoding="utf-8") as f:
    eng_data = f.read().splitlines()

print(f"한국어 원본 데이터 개수: {len(kor_data)}")
print(f"영어 원본 데이터 개수: {len(eng_data)}")
print("\n[데이터 샘플 확인]")
print(f"KOR: {kor_data[0]}")
print(f"ENG: {eng_data[0]}")


데이터 다운로드 중...
다운로드 완료!
압축 해제 완료!


/tmp/ipykernel_1497/3591264993.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dataset_dir)


한국어 원본 데이터 개수: 94123
영어 원본 데이터 개수: 94123

[데이터 샘플 확인]
KOR: 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
ENG: Much of personal computing is about "can you top this?"


---
## Step 2. 데이터 정제

1. `set` 데이터형을 활용하여 병렬 쌍이 흐트러지지 않게 중복 데이터를 제거하고 `cleaned_corpus`에 저장합니다.
2. 한글과 영문에 각각 적용할 수 있는 정규식을 기반으로 `preprocess_sentence()` 함수를 정의합니다.
3. 타겟 언어인 영문에는 `<start>` 토큰과 `<end>` 토큰을 추가하고 `split()`으로 토큰화하며, 한글 토큰화는 KoNLPy의 `Mecab`을 사용합니다.
4. 토큰의 길이가 40 이하인 데이터를 선별하여 `eng_corpus`와 `kor_corpus`를 구축합니다.

In [5]:
# 1. set을 활용한 중복 데이터 제거
# 병렬 데이터의 정렬 순서를 무너뜨리지 않고 (KOR, ENG) 쌍을 하나로 묶어 중복 쌍을 제거합니다.
cleaned_corpus = list(set(zip(kor_data, eng_data)))
cleaned_kor, cleaned_eng = zip(*cleaned_corpus)

print(f"중복 제거 후 데이터 개수: {len(cleaned_kor)}")


중복 제거 후 데이터 개수: 78968


In [6]:
# 2. 정규식을 활용한 문장 정제 함수
def preprocess_sentence(sentence, s_type="kor"):
    sentence = sentence.lower().strip()

    # 단어와 구두점 사이에 공백을 추가하여 구두점을 독립된 토큰으로 분리
    # 예: "hello." -> "hello ." (구두점이 단어에 붙어 희귀 단어가 되는 것을 방지)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)

    if s_type == "kor":
        # 한국어: 한글, 영문, 숫자, 구두점 제외한 불필요한 특수문자 제거
        sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    else:
        # 영어: 영문, 구두점 제외한 문자 제거
        sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)

    sentence = sentence.strip()
    return sentence

print("전처리 함수 정의 완료!")


전처리 함수 정의 완료!


In [7]:
# 3. 형태소 분석 및 토큰화, 시퀀스 길이 필터링
try:
    mecab = Mecab()
except:
    print("Mecab 로드 실패. 대체 토크나이저를 사용합니다.")
    class BaseTokenizer:
        def morphs(self, text):
            return text.split()
    mecab = BaseTokenizer()

kor_corpus = []
eng_corpus = []

for k, e in tqdm(zip(cleaned_kor, cleaned_eng), total=len(cleaned_kor), desc="데이터 정제 및 토큰화"):
    k_pre = preprocess_sentence(k, s_type="kor")
    e_pre = preprocess_sentence(e, s_type="eng")

    # 한국어는 Mecab을 통해 형태소 단위로 분리 (BPE 사전 분절 역할)
    k_tokens = mecab.morphs(k_pre)
    # 디코더의 입력 및 레이블 생성을 위해 영어 시퀀스 앞뒤에 <start>와 <end> 토큰 추가
    e_tokens = ["<start>"] + e_pre.split() + ["<end>"]

    # Attention 메모리 부족(OOM) 방지 및 연산 효율을 위해 최대 길이를 40 이하로 제한
    if len(k_tokens) <= 40 and len(e_tokens) <= 40:
        kor_corpus.append(k_tokens)
        eng_corpus.append(e_tokens)

print(f"\n길이 40 이하 필터링 후 최종 데이터 개수: {len(kor_corpus)}")
print(f"[한국어 토큰 샘플]: {kor_corpus[0]}")
print(f"[영어 토큰 샘플]: {eng_corpus[0]}")


Mecab 로드 실패. 대체 토크나이저를 사용합니다.


데이터 정제 및 토큰화: 100%|██████████| 78968/78968 [00:04<00:00, 18887.44it/s]


길이 40 이하 필터링 후 최종 데이터 개수: 70167
[한국어 토큰 샘플]: ['이밖에', '경찰은', '15일', '바그다드에서', '30발의', '총탄을', '맞은', '시신', '몇', '구를', '바그다드에서', '발견했다', '.']
[영어 토큰 샘플]: ['<start>', 'separately', ',', 'police', 'said', 'bullet', 'riddled', 'bodies', 'were', 'found', 'by', 'iraqi', 'police', 'across', 'the', 'capital', 'on', 'sunday', '.', '<end>']


---
## Step 3. 데이터 토큰화

`tokenize()` 함수를 정의하여 데이터를 텐서로 변환하고 각각의 토크나이저를 얻습니다.
단어의 수는 최소 10,000 이상으로 설정합니다. (본 실습에서는 `12,000`으로 설정)

❗ **주의**: 난이도에 비해 데이터가 많지 않아 훈련 데이터와 검증 데이터를 따로 나누지 않고 전체를 훈련에 사용합니다.

In [8]:
# BPE 기반 Subword Tokenizer 및 기존 인터페이스 호환 래퍼 클래스 정의
class BPETokenizerWrapper:
    def __init__(self, num_words=12000, pad_token="<pad>", unk_token="<unk>", start_token="<start>", end_token="<end>"):
        self.num_words = num_words
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.start_token = start_token
        self.end_token = end_token

        # Hugging Face BPE 토크나이저 초기화
        self.tokenizer = Tokenizer(BPE(unk_token=unk_token))
        self.tokenizer.pre_tokenizer = Whitespace()

        # 호환성을 위한 딕셔너리 구조
        self.word_index = {}
        self.index_word = {}

    def fit_on_texts(self, texts):
        # texts는 토큰 리스트의 리스트
        # BPE 학습을 위해 문자열 리스트로 변환
        text_strings = [" ".join(seq) for seq in texts]

        # 스페셜 토큰 정의 (순서대로 0: <pad>, 1: <unk>, 2: <start>, 3: <end>)
        special_tokens = [self.pad_token, self.unk_token, self.start_token, self.end_token]
        trainer = BpeTrainer(special_tokens=special_tokens, vocab_size=self.num_words)

        self.tokenizer.train_from_iterator(text_strings, trainer=trainer)

        # 호환성 속성 채우기
        vocab = self.tokenizer.get_vocab()
        for word, idx in vocab.items():
            self.word_index[word] = idx
            self.index_word[idx] = word

        # 스페셜 토큰 명시적 보장
        for idx, token in enumerate(special_tokens):
            self.word_index[token] = idx
            self.index_word[idx] = token

    def texts_to_sequences(self, texts):
        text_strings = [" ".join(seq) for seq in texts]
        encoded = self.tokenizer.encode_batch(text_strings)
        return [enc.ids for enc in encoded]

    def pad_sequences(self, sequences, maxlen=40, padding="post"):
        pad_id = self.word_index[self.pad_token]
        padded = []
        for seq in sequences:
            if len(seq) < maxlen:
                if padding == "post":
                    seq = list(seq) + [pad_id] * (maxlen - len(seq))
                else:
                    seq = [pad_id] * (maxlen - len(seq)) + list(seq)
            else:
                seq = list(seq)[:maxlen]
            padded.append(seq)
        return torch.tensor(padded, dtype=torch.long)

    def encode(self, text_list):
        text_string = " ".join(text_list)
        return self.tokenizer.encode(text_string).ids

    def decode(self, token_ids):
        # BPE 토큰을 문자열로 디코딩한 후 리스트로 반환하여 기존 시각화 함수와 호환성 유지
        return [self.index_word.get(token_id, self.unk_token) for token_id in token_ids]

    def __len__(self):
        return self.tokenizer.get_vocab_size()

def tokenize_bpe(corpus, num_words=12000):
    tokenizer = BPETokenizerWrapper(num_words=num_words)
    tokenizer.fit_on_texts(corpus)
    sequences = tokenizer.texts_to_sequences(corpus)
    tensor = tokenizer.pad_sequences(sequences, maxlen=40)
    return tensor, tokenizer

# 단어 수 12,000 설정 (최소 10,000 이상)
VOCAB_SIZE = 12000
enc_tensor, enc_tokenizer = tokenize_bpe(kor_corpus, num_words=VOCAB_SIZE)
dec_tensor, dec_tokenizer = tokenize_bpe(eng_corpus, num_words=VOCAB_SIZE)

print(f"Korean Tensor Shape: {enc_tensor.shape}")
print(f"English Tensor Shape: {dec_tensor.shape}")
print(f"Korean Vocab Size: {len(enc_tokenizer)}")
print(f"English Vocab Size: {len(dec_tokenizer)}")


Korean Tensor Shape: torch.Size([70167, 40])
English Tensor Shape: torch.Size([70167, 40])
Korean Vocab Size: 12000
English Vocab Size: 12000


In [9]:
# 교사 강요(Teacher Forcing)를 위한 타겟 텐서 슬라이싱
class NMTDataset(Dataset):
    def __init__(self, src_tensor, trg_tensor):
        self.src_tensor = src_tensor
        # 디코더 입력(trg_input): 교사 강요 학습을 위해 마지막 토큰(<end> 또는 패딩)을 제외합니다.
        # 예: [<start>, i, am, a, student, <end>] -> [<start>, i, am, a, student]
        self.trg_input = trg_tensor[:, :-1]

        # 디코더 정답(trg_label): 예측해야 할 대상이므로 첫 번째 토큰(<start>)을 제외합니다.
        # 예: [<start>, i, am, a, student, <end>] -> [i, am, a, student, <end>]
        self.trg_label = trg_tensor[:, 1:]

    def __len__(self):
        return len(self.src_tensor)

    def __getitem__(self, idx):
        return self.src_tensor[idx], self.trg_input[idx], self.trg_label[idx]

BATCH_SIZE = 64
train_dataset = NMTDataset(enc_tensor, dec_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

for src, trg_input, trg_label in train_loader:
    print("DataLoader 배치 형태 확인:")
    print(f"src shape: {src.shape}, trg_input shape: {trg_input.shape}, trg_label shape: {trg_label.shape}")
    break


DataLoader 배치 형태 확인:
src shape: torch.Size([64, 40]), trg_input shape: torch.Size([64, 39]), trg_label shape: torch.Size([64, 39])


---
## Step 4. 모델 설계

한국어를 영어로 번역하기 위한 Bahdanau Attention 기반 Seq2seq 모델을 설계합니다.
- **Embedding Size**: 256
- **Hidden Size**: 512

In [10]:
# 1. Bahdanau Attention (Additive Attention) - 패딩 마스킹 적용
# 디코더가 단어를 생성할 때마다 인코더의 전체 출력 중 어느 부분에 집중(Attention)해야 할지 가중치를 계산합니다.
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # 인코더가 양방향(Bidirectional)이므로 encoder_outputs의 차원은 hidden_dim * 2입니다.
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        # hidden: 디코더의 현재 은닉 상태 (batch_size, hidden_dim)
        # encoder_outputs: 인코더의 모든 타임스텝 출력 (src_len, batch_size, hidden_dim * 2)
        src_len = encoder_outputs.shape[0]
        # 어텐션 에너지 계산을 위해 차원 및 시퀀스 길이 일치 작업
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim * 2)

        # 어텐션 점수(Score) 계산: score(s_t, h_i) = v^T * tanh(W1*h_i + W2*s_t)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch_size, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)

        # 어텐션 패딩 마스킹(Padding Masking): 패딩 위치의 어텐션 에너지를 극소값(-1e9)으로 채워 소프트맥스 가중치가 0이 되게 만듭니다.
        if mask is not None:
            attention = attention.masked_fill(mask, -1e9)

        # 소프트맥스(Softmax)를 거쳐 시퀀스 합이 1이 되는 어텐션 가중치 분포 생성
        return nn.functional.softmax(attention, dim=1)  # (batch_size, src_len)

# 2. Encoder - 양방향(Bidirectional) GRU 적용
# 소스 시퀀스를 임베딩하고 양방향 GRU를 통과시켜 전후 맥락이 파악된 각 타임스텝의 출력과 최종 은닉 상태를 생성합니다.
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, bidirectional=True)
        # 양방향 은닉 상태(hidden_dim * 2)를 디코더의 단방향 은닉 상태(hidden_dim)로 투영
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src):
        # src : (src_len, batch_size)
        embedded = self.embedding(src)  # embedded : (src_len, batch_size, emb_dim)
        outputs, hidden = self.rnn(embedded)  # outputs : (src_len, batch_size, hidden_dim * 2), hidden: (2, batch_size, hidden_dim)

        # 정방향(forward)과 역방향(backward)의 마지막 은닉 상태를 결합(concat)하여 Linear 통과
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)  # (batch_size, hidden_dim * 2)
        hidden = torch.tanh(self.fc_hidden(hidden)).unsqueeze(0)  # (1, batch_size, hidden_dim)
        return outputs, hidden

# 3. Decoder
# 어텐션을 통해 컨텍스트 벡터를 생성하고, 이를 현재 단어 임베딩 및 이전 은닉 상태와 결합하여 다음 단어를 예측합니다.
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super(Decoder, self).__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        # GRU 출력(hidden_dim) + 양방향 컨텍스트 벡터(hidden_dim * 2)를 합쳐서(concat) 단어 사전 크기(output_dim)로 투영
        self.fc_out = nn.Linear(hidden_dim + hidden_dim * 2, output_dim)

    def forward(self, input, hidden, encoder_outputs, mask=None):
        # input : (batch_size,) (단일 타임스텝 입력)
        # hidden : (1, batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim * 2)
        input = input.unsqueeze(0)  # input : (1, batch_size)
        embedded = self.embedding(input)  # embedded : (1, batch_size, emb_dim)

        # 어텐션 가중치 a 계산 (마스크 전달)
        a = self.attention(hidden[-1], encoder_outputs, mask)  # a : (batch_size, src_len)

        # 어텐션 가중치와 인코더 출력의 배치 행렬곱(bmm)을 통해 컨텍스트 벡터 계산
        a = a.unsqueeze(1)  # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # encoder_outputs : (batch_size, src_len, hidden_dim * 2)
        context = torch.bmm(a, encoder_outputs)  # context : (batch_size, 1, hidden_dim * 2)
        context = context.permute(1, 0, 2)  # context : (1, batch_size, hidden_dim * 2)

        # 디코더 GRU 연산
        output, hidden = self.rnn(embedded, hidden)

        # 디코더 출력과 컨텍스트 벡터를 결합(concat)하여 최종 예측값 생성
        output = output.squeeze(0)  # output : (batch_size, hidden_dim)
        context = context.squeeze(0)  # context : (batch_size, hidden_dim * 2)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, a.squeeze(1)

# 4. Seq2SeqAttention 통합 모델 - Scheduled Sampling 및 Mask 생성 적용
# 인코더와 디코더를 하나로 연결하며, 학습 시 교사 강요 비율을 조절하고 추론 시 탐욕적 탐색 모드를 지원합니다.
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device, pad_id=0):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.pad_id = pad_id

    def forward(self, src, trg=None, teacher_forcing_ratio=0.5, max_len=40, bos_id=2, eos_id=3):
        # src: (src_len, batch_size)
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = []
        attentions = []

        # 패딩 마스크 생성: src에서 pad_id 위치를 True로 설정 (batch_size, src_len)
        mask = (src == self.pad_id).permute(1, 0)

        # 인코더를 거쳐 소스 문장의 전체 타임스텝 표현과 마지막 은닉 상태 생성
        encoder_outputs, hidden = self.encoder(src)

        if trg is not None:
            # [학습 모드 - Scheduled Sampling (Teacher Forcing 비율 조절)]
            # t=0일 때는 <start> 토큰(trg[0])을 넣고, 이후 스텝부터는 teacher_forcing_ratio 확률에 따라 정답 또는 이전 예측값을 주입합니다.
            input = trg[0]
            for t in range(0, trg.shape[0]):
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs, mask)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))

                top1 = output.argmax(1)
                # t+1번 스텝을 위한 입력 설정
                if t + 1 < trg.shape[0]:
                    input = trg[t+1] if random.random() < teacher_forcing_ratio else top1
        else:
            # [추론 모드 - 탐욕적 탐색(Greedy Search)]
            # 타겟 문장이 없으므로 최초 입력으로 <start>(bos_id)를 넣고, 매 스텝 예측된 최고 확률 단어를 다음 스텝의 입력으로 사용합니다.
            input = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs, mask)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))
                # 최고 확률 단어 선택
                top1 = output.argmax(1)
                input = top1

                # 모든 배치가 <end>(eos_id)를 생성하면 조기 종료
                finished |= (top1 == eos_id)
                if finished.all():
                    break

        outputs = torch.cat(outputs, dim=0)  # (trg_len, batch_size, output_dim)
        attentions = torch.cat(attentions, dim=0)  # (trg_len, batch_size, src_len)

        return outputs, attentions


In [11]:
# 모델 하이퍼파라미터 정의 및 인스턴스 생성
input_dim = len(enc_tokenizer)
output_dim = len(dec_tokenizer)
emb_dim = 256
hid_dim = 512
pad_id = dec_tokenizer.word_index[dec_tokenizer.pad_token]

# 각 모듈 생성 및 디바이스(GPU) 할당
encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
attention = BahdanauAttention(hid_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, attention).to(device)
model = Seq2SeqAttention(encoder, decoder, device, pad_id=pad_id).to(device)

print(model)


Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(12000, 256)
    (rnn): GRU(256, 512, bidirectional=True)
    (fc_hidden): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=1024, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(12000, 256)
    (rnn): GRU(256, 512)
    (fc_out): Linear(in_features=1536, out_features=12000, bias=True)
  )
)


---
## Step 5. 훈련하기 및 결과 시각화

훈련 코드는 `eval_step()` 없이 진행되며, 매 에포크 스텝마다 제시된 예문에 대한 번역을 생성하여 품질을 확인합니다.
학습이 완료된 후 최종 번역 결과와 함께 Attention Map을 시각화합니다.

### [예문]
- **K1)** 오바마는 대통령이다.
- **K2)** 시민들은 도시 속에 산다.
- **K3)** 커피는 필요 없다.
- **K4)** 일곱 명의 사망자가 발생했다.

In [12]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
pad_id = dec_tokenizer.word_index[dec_tokenizer.pad_token]
# ignore_index=pad_id를 설정하여 패딩(<pad>) 토큰에 대한 예측은 손실(Loss) 계산에서 완전히 제외시킴으로써 학습의 정확도를 높입니다.
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
# 학습률 스케줄러: 손실 감소가 정체될 때 학습률을 0.5배로 감소
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# 학습 루프 함수
def train_step(model, data_loader, optimizer, criterion, epoch, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)

    for src, trg_input, trg_label in progress_bar:
        # PyTorch RNN 계열 모델의 기본 기대 입력 차원인 (시퀀스 길이, 배치 사이즈)로 맞추기 위해 permute(1, 0) 실행
        src = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)

        optimizer.zero_grad()

        # 순전파 계산 (학습 모드이므로 trg_input 및 teacher_forcing_ratio 전달)
        outputs,_ = model(src, trg=trg_input, teacher_forcing_ratio=teacher_forcing_ratio)

        # 손실 계산을 위해 출력과 레이블 텐서를 2차원과 1차원으로 평탄화(Flatten)
        # outputs: (배치 크기 * 시퀀스 길이, 단어 사전 크기)
        outputs = outputs.reshape(-1, outputs.shape[-1])
        # trg_label: (배치 크기 * 시퀀스 길이)
        trg_label = trg_label.reshape(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()

        # 기울기 폭발(Gradient Exploding) 방지: 그레디언트 노름(Norm)이 1을 넘지 않도록 클리핑
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    return epoch_loss / len(data_loader)

In [13]:
# 모델 추론 및 Attention Map 시각화
import matplotlib.font_manager as fm

# 단일 문장에 대한 번역 추론을 수행하는 함수
def evaluate(sentence, model, enc_tokenizer, dec_tokenizer, max_len=40):
    model.eval()  # 평가 모드 전환
    pre_sentence = preprocess_sentence(sentence, s_type="kor")
    # Mecab 형태소 분석 후 BPE 인코딩 수행
    tokens = mecab.morphs(pre_sentence)
    src_ids = enc_tokenizer.encode(tokens)

    # 시각화 시 어텐션 맵 차원 정확성을 위해 실제 BPE 분절 토큰 리스트 추출
    bpe_tokens = enc_tokenizer.decode(src_ids)

    src_ids = src_ids[:max_len]
    bpe_tokens = bpe_tokens[:max_len]

    # 시퀀스 길이(max_len)를 맞추기 위한 패딩 추가
    src_ids = src_ids + [enc_tokenizer.word_index[enc_tokenizer.pad_token]] * (max_len - len(src_ids))
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)  # (max_len, batch_size=1)

    bos_id = dec_tokenizer.word_index["<start>"]
    eos_id = dec_tokenizer.word_index["<end>"]

    # 추론 시에는 불필요한 그레디언트 연산을 비활성화(no_grad)하여 속도를 높이고 메모리를 절약
    with torch.no_grad():
        # trg=None으로 전달하여 모델이 자기 회귀(Auto-regressive) 방식으로 단어를 생성하도록 유도
        outputs, attentions = model(src_tensor, trg=None, max_len=max_len, bos_id=bos_id, eos_id=eos_id)

    # 예측된 단어 ID를 가져오고 문자열로 디코딩
    result_ids = outputs.argmax(2).squeeze(1).cpu().numpy()
    result = dec_tokenizer.decode(result_ids)
    # 생성된 문장 중 <end> 토큰 이후의 결과는 제거하여 최종 번역 문장만 도출
    if "<end>" in result: result = result[:result.index("<end>")]
    return result, bpe_tokens, attentions.squeeze(1).cpu().numpy()

# 번역된 각 단어가 원본 문장의 어떤 단어에 집중(Attention)했는지 보여주는 히트맵 생성
def plot_attention(attention, sentence_tokens, predicted_tokens, epoch, example_idx):
    font_path = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
    fp = fm.FontProperties(fname=font_path, size=12)

    fig, ax = plt.subplots(figsize=(8, 8))
    # 어텐션 가중치 행렬을 2차원 매트릭스로 렌더링
    ax.matshow(attention, cmap='viridis')

    ax.set_xticks(range(len(sentence_tokens)))
    # x축: 원본 한국어 토큰 (한글 폰트 적용)
    ax.set_xticklabels(sentence_tokens, fontproperties=fp, rotation=90)

    ax.set_yticks(range(len(predicted_tokens)))
    # y축: 번역된 영어 토큰
    ax.set_yticklabels(predicted_tokens, size=12)

    plt.title(f"Epoch {epoch+1} - Example {example_idx}", fontsize=14)
    plt.tight_layout()
    plt.show()

def translate_and_show(sentence, model, enc_tokenizer, dec_tokenizer, epoch, example_idx, show_plot=False):
    result, bpe_tokens, attention = evaluate(sentence, model, enc_tokenizer, dec_tokenizer, max_len=40)
    pred_str = " ".join(result) + " <end>"
    print(f"K{example_idx}) {sentence}")
    print(f"E{example_idx}) {pred_str}\n")
    if show_plot:
        # 실제 유효한 문장 길이만큼만 어텐션 행렬을 슬라이싱하여 직관적인 히트맵 출력
        attention = attention[:len(result), :len(bpe_tokens)]
        plot_attention(attention, bpe_tokens, result, epoch, example_idx)


In [ ]:
%%time

# 에포크 수를 30회로 상향 조정하여 모델 수렴 강화
EPOCHS = 30
example_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

# 전체 에포크 동안 모델 학습 및 단계별 번역 품질 모니터링
for epoch in range(EPOCHS):
    # 학습 후반부로 갈수록 교사 강요 비율을 점진적으로 낮춤 (0.7 -> 0.3)
    tf_ratio = max(0.3, 0.7 - (epoch / EPOCHS) * 0.4)
    train_loss = train_step(model, train_loader, optimizer, criterion, epoch, teacher_forcing_ratio=tf_ratio)
    scheduler.step(train_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, TF Ratio: {tf_ratio:.2f}\n")

    print("=== 매 스텝 번역 예문 결과 ===")
    # 마지막 에포크에서 최종 번역 결과와 함께 Attention Map을 시각화하여 모델의 학습 해석 가능성(Interpretability) 점검
    show_plot = (epoch == EPOCHS - 1)
    for idx, sentence in enumerate(example_sentences, start=1):
        translate_and_show(sentence, model, enc_tokenizer, dec_tokenizer, epoch, idx, show_plot=show_plot)
    print("-" * 50)


Colab 환경에서 epoch 30 으로 모델을 설정하였으나 런타임 할당으로 인해 epoch 29 에서 중단되는 사태가 발생하였다.

다행히 epoch 27 까지의 내용은 저장해놨기에 아래에 표기하였습니다.

그리고 셀이 멈추면서 각 값들이 초기화 되었기에 시각화가 불가능한 점 양해부탁드립니다.

___

Epoch 1: 100%|██████████| 1097/1097 [06:36<00:00,  2.76it/s, loss=5.33]
Epoch 1/30, Train Loss: 5.9052, TF Ratio: 0.70

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is obama . <end>

K2) 시민들은 도시 속에 산다.
E2) the city is a . <end>

K3) 커피는 필요 없다.
E3) the is not . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the two people were killed . <end>

--------------------------------------------------
Epoch 2: 100%|██████████| 1097/1097 [06:42<00:00,  2.73it/s, loss=4.5]
Epoch 2/30, Train Loss: 4.8918, TF Ratio: 0.69

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president elect . <end>

K2) 시민들은 도시 속에 산다.
E2) the city is the city of the city s cities . <end>

K3) 커피는 필요 없다.
E3) caffeine s coffee , coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the blast occurred . <end>

--------------------------------------------------
Epoch 3: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=4.61]
Epoch 3/30, Train Loss: 4.2668, TF Ratio: 0.67

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the urban ization is a city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee coffee coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were killed . <end>

--------------------------------------------------
Epoch 4: 100%|██████████| 1097/1097 [06:41<00:00,  2.74it/s, loss=4.19]
Epoch 4/30, Train Loss: 3.7772, TF Ratio: 0.66

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president s . <end>

K2) 시민들은 도시 속에 산다.
E2) it s urban cities urban cities . <end>

K3) 커피는 필요 없다.
E3) there is no coffee coffee coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the deaths occurred . <end>

--------------------------------------------------
Epoch 5: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.75]
Epoch 5/30, Train Loss: 3.4332, TF Ratio: 0.65

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president . <end>

K2) 시민들은 도시 속에 산다.
E2) the urban city city city of urban ization . <end>

K3) 커피는 필요 없다.
E3) there was no coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths were killed in the death . <end>

--------------------------------------------------
Epoch 6: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.17]
Epoch 6/30, Train Loss: 3.1618, TF Ratio: 0.63

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president . <end>

K2) 시민들은 도시 속에 산다.
E2) fur y acid hit urban urban city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths were killed in the . <end>

--------------------------------------------------
Epoch 7: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.06]
Epoch 7/30, Train Loss: 2.9775, TF Ratio: 0.62

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) the urban areas in urban areas . <end>

K3) 커피는 필요 없다.
E3) there hasn t coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were killed . <end>

--------------------------------------------------
Epoch 8: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.75]
Epoch 8/30, Train Loss: 2.8474, TF Ratio: 0.61

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) there s urban areas city city cities . <end>

K3) 커피는 필요 없다.
E3) there was no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were killed in the blast occurred . <end>

--------------------------------------------------
Epoch 9: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.05]
Epoch 9/30, Train Loss: 2.7541, TF Ratio: 0.59

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president . <end>

K2) 시민들은 도시 속에 산다.
E2) there s a friendly cities in the city city city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were deaths were killed in the blast . <end>

--------------------------------------------------
Epoch 10: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.02]
Epoch 10/30, Train Loss: 2.6477, TF Ratio: 0.58

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) there s urban city city of urban spraw . <end>

K3) 커피는 필요 없다.
E3) there is no coffee coffee coffee coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were deaths in the blast occurred . <end>

--------------------------------------------------
Epoch 11: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=3.22]
Epoch 11/30, Train Loss: 2.5877, TF Ratio: 0.57

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) president obama is the president . <end>

K2) 시민들은 도시 속에 산다.
E2) there s cities passed urban cities . <end>

K3) 커피는 필요 없다.
E3) there is no coffee coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths deaths occurred in the seven seven deaths . <end>

--------------------------------------------------
Epoch 12: 100%|██████████| 1097/1097 [06:43<00:00,  2.72it/s, loss=2.49]
Epoch 12/30, Train Loss: 2.5479, TF Ratio: 0.55

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president is a . <end>

K2) 시민들은 도시 속에 산다.
E2) neither city s dae gu is . cities city . <end>

K3) 커피는 필요 없다.
E3) but no need for coffee coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths were deaths . <end>

--------------------------------------------------
Epoch 13: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.44]
Epoch 13/30, Train Loss: 2.4869, TF Ratio: 0.54

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) president obama is the presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities in cities cities cities in cities . <end>

K3) 커피는 필요 없다.
E3) but no coffee was coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths deaths were deaths . <end>

--------------------------------------------------
Epoch 14: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.72]
Epoch 14/30, Train Loss: 2.4683, TF Ratio: 0.53

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities cities passed urban cities . <end>

K3) 커피는 필요 없다.
E3) coffee was not a coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were dead . <end>

--------------------------------------------------
Epoch 15: 100%|██████████| 1097/1097 [06:42<00:00,  2.72it/s, loss=3.19]
Epoch 15/30, Train Loss: 2.4394, TF Ratio: 0.51

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the president . <end>

K2) 시민들은 도시 속에 산다.
E2) the boys are cities in cities city city . <end>

K3) 커피는 필요 없다.
E3) but there s no coffee for a coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were dead , <end>

--------------------------------------------------
Epoch 16: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.41]
Epoch 16/30, Train Loss: 2.4274, TF Ratio: 0.50

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is the presidential president . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities passed the city city city urban city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven deaths were deaths . <end>

--------------------------------------------------
Epoch 17: 100%|██████████| 1097/1097 [06:40<00:00,  2.74it/s, loss=2.65]
Epoch 17/30, Train Loss: 2.4151, TF Ratio: 0.49

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president . <end>

K2) 시민들은 도시 속에 산다.
E2) they ve cities in the city city city city . <end>

K3) 커피는 필요 없다.
E3) there was no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were abducted in the il s . <end>

--------------------------------------------------
Epoch 18: 100%|██████████| 1097/1097 [06:40<00:00,  2.74it/s, loss=2.32]
Epoch 18/30, Train Loss: 2.4252, TF Ratio: 0.47

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the city was passed city city city city city city city . <end>

K3) 커피는 필요 없다.
E3) no coffee was coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were deaths deaths . <end>

--------------------------------------------------
Epoch 19: 100%|██████████| 1097/1097 [06:40<00:00,  2.74it/s, loss=2.54]
Epoch 19/30, Train Loss: 2.4434, TF Ratio: 0.46

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama president is presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) ex s urban city in urban city . <end>

K3) 커피는 필요 없다.
E3) but there need no coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven were deaths were in . <end>

--------------------------------------------------
Epoch 20: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.93]
Epoch 20/30, Train Loss: 2.4421, TF Ratio: 0.45

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is a presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities is city city city city city city . <end>

K3) 커피는 필요 없다.
E3) there need a coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the seven people were were dead . <end>

--------------------------------------------------
Epoch 21: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=2.15]
Epoch 21/30, Train Loss: 2.1723, TF Ratio: 0.43

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is a presidential . <end>

K2) 시민들은 도시 속에 산다.
E2) they city in cities city city city . <end>

K3) 커피는 필요 없다.
E3) there is no need for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the seven deaths were were . <end>

--------------------------------------------------
Epoch 22: 100%|██████████| 1097/1097 [06:40<00:00,  2.74it/s, loss=1.63]
Epoch 22/30, Train Loss: 1.8514, TF Ratio: 0.42

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the boys city acid in urban areas . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the deaths were were in . <end>

--------------------------------------------------
Epoch 23: 100%|██████████| 1097/1097 [06:42<00:00,  2.72it/s, loss=2.36]
Epoch 23/30, Train Loss: 1.7291, TF Ratio: 0.41

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) a city city city city city city city urban city city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the seven people were abducted . <end>

--------------------------------------------------
Epoch 24: 100%|██████████| 1097/1097 [06:42<00:00,  2.72it/s, loss=1.71]
Epoch 24/30, Train Loss: 1.6718, TF Ratio: 0.39

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the city s rural city city city urban city city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the deaths were were in . <end>

--------------------------------------------------
Epoch 25: 100%|██████████| 1097/1097 [06:39<00:00,  2.74it/s, loss=1.45]
Epoch 25/30, Train Loss: 1.6459, TF Ratio: 0.38

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) a city cities passed the city city city urban city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) seven people were were . <end>

--------------------------------------------------
Epoch 26: 100%|██████████| 1097/1097 [06:40<00:00,  2.74it/s, loss=1.77]
Epoch 26/30, Train Loss: 1.6137, TF Ratio: 0.37

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities in the city urban city city city . <end>

K3) 커피는 필요 없다.
E3) there is no coffee for coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the were people were abducted . <end>

--------------------------------------------------
Epoch 27: 100%|██████████| 1097/1097 [06:41<00:00,  2.73it/s, loss=1.92]
Epoch 27/30, Train Loss: 1.6100, TF Ratio: 0.35

=== 매 스텝 번역 예문 결과 ===
K1) 오바마는 대통령이다.
E1) obama is president president . <end>

K2) 시민들은 도시 속에 산다.
E2) the cities in cities city city city urban city . <end>

K3) 커피는 필요 없다.
E3) no need to coffee . <end>

K4) 일곱 명의 사망자가 발생했다.
E4) the were fatal in the center . <end>

--------------------------------------------------

# 💡 프로젝트 회고 및 종합 평가 (Retrospective)

본 프로젝트는 RNN 기반의 양방향 GRU(Bidirectional GRU)와 Bahdanau Attention 메커니즘을 결합하여 한국어-영어 기계 번역(NMT) 모델을 처음부터 설계하고 학습시키는 전 과정을 다루었습니다. 30 에포크(Epoch) 동안의 실시간 학습 지표와 번역 생성 품질의 변화를 다각도로 분석하며 얻은 성과와 향후 발전 방향을 정리합니다.

---

## 1. 주요 성과 및 성공 요인

### 1.1. 양방향 인코더 도입을 통한 어순 극복
- 한국어(SOV)와 영어(SVO)는 어순이 완전히 달라 단방향 RNN 구조로는 번역 품질에 명확한 한계가 존재했습니다.
- `Encoder`의 GRU를 양방향(`bidirectional=True`)으로 설정하고 은닉 상태 투영 레이어(`fc_hidden`)를 결합함으로써, 문장 끝에 위치한 한국어 서술어의 의미를 디코더 초반(영어 동사 생성 시점)에 성공적으로 전달할 수 있었습니다.

### 1.2. BPE Subword 토크나이저를 통한 OOV 극복 및 희귀 단어 번역
- 기존 빈도수 기반 단어 사전의 한계로 인해 뉴스 코퍼스 외의 어휘가 `<unk>`로 대체되는 문제를 해결하기 위해 Hugging Face `tokenizers`의 BPE 모델을 연동했습니다.
- 그 결과, 뉴스 데이터셋에서 드물게 등장하는 **"커피는 필요 없다."**와 같은 일상 예문에서도 `coffee`, `need` 등의 핵심 단어를 정확히 포착하여, Epoch 21 이후 `there is no need for coffee.` / `no need to coffee.`라는 완벽하고 자연스러운 영문 번역을 이뤄냈습니다.

### 1.3. 스케줄러와 교사 강요 비율 스케줄링(Scheduled Sampling)의 완벽한 조화
- **정체기 극복**: Epoch 11~20 구간에서 Train Loss가 2.44 부근에서 정체(Plateau)되었으나, 도입한 `ReduceLROnPlateau` 스케줄러가 개입하여 학습률을 자동으로 조정함으로써 Epoch 28 기준 **1.61**까지 극적인 손실 감소와 미세 조정(Fine-tuning)을 이끌어냈습니다.
- **노출 편향(Exposure Bias) 제거**: 매 에포크마다 교사 강요 비율을 점진적으로 낮춤(0.7 $\rightarrow$ 0.3)으로써, 디코더가 자신의 이전 예측값(`argmax`)을 다음 입력으로 주입하는 훈련을 안정적으로 수행하여 실제 추론 시의 안정성을 극대화했습니다.

---

## 2. 한계점 및 향후 개선 방향 (Next Steps)

### 2.1. 탐욕적 탐색(Greedy Search)으로 인한 단어 반복 현상
- **관찰 양상**: 후반부 에포크에서 일부 문장 생성 시 `president president`, `city city`와 같이 동일한 단어가 반복되는 현상이 관찰되었습니다.
- **원인 및 해결책**: 이는 현재 추론 함수인 `evaluate`가 매 타임스텝마다 확률이 가장 높은 단어 단 하나만 선택하는 탐욕적 탐색(Greedy Search)을 취하고 있기 때문에, 직전 생성 단어의 조건부 확률에 강하게 얽매이는 현상입니다.
- **향후 과제**: 디코더 추론 시 후보군 3~5개를 동시에 탐색하는 **빔 서치(Beam Search)** 및 이미 생성된 단어의 로짓을 낮추는 **반복 패널티(Repetition Penalty)** 로직을 도입하면 단어 반복이 완벽히 사라지고 한층 더 완성도 높은 문장 생성이 가능할 것입니다.

### 2.2. 트랜스포머(Transformer) 아키텍처로의 확장
- 본 프로젝트를 통해 순차 연산(RNN) 기반 어텐션 메커니즘의 훌륭한 성능과 내부 동작 원리(Attention Map 히트맵)를 충분히 검증했습니다.
- 향후에는 순차 연산을 완전히 제거하고 병렬 처리를 극대화한 **트랜스포머(Transformer: Self-Attention & Multi-Head Attention)** 아키텍처로 확장하여, 더 긴 시퀀스와 대규모 데이터셋에 대한 번역 속도 및 품질을 비교 분석하는 것을 다음 목표로 삼을 것입니다.
